# 02 — Feature Engineering

This notebook demonstrates the feature engineering pipeline:
1. Temporal features (cyclical encoding, peak flags)
2. Spatial features (distance, bearing)
3. Contextual features (supply-demand, interactions)
4. Full pipeline orchestration
5. Feature importance analysis

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

from src.data.loader import load_and_prepare
from src.data.splitter import split_by_time
from src.features.temporal import add_temporal_features
from src.features.spatial import add_spatial_features
from src.features.contextual import (
    add_supply_demand_features,
    add_order_features,
    add_interaction_features,
)
from src.features.pipeline import FeaturePipeline

In [ ]:
df = load_and_prepare("../data/raw/delivery_data.csv")
print(f"Raw dataset: {df.shape}")
df.head(3)

## 1. Temporal Features

In [ ]:
df_temp = add_temporal_features(df.copy())
temporal_cols = [c for c in df_temp.columns if c not in df.columns]
print(f"Temporal features added: {temporal_cols}")
df_temp[temporal_cols].head(10)

In [ ]:
# Visualize cyclical encoding
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
sample = df_temp.drop_duplicates(subset=["hour"]).sort_values("hour")
ax.scatter(sample["hour_sin"], sample["hour_cos"], c=sample["hour"], cmap="hsv", s=100)
for _, row in sample.iterrows():
    ax.annotate(f"h={int(row['hour'])}", (row["hour_sin"], row["hour_cos"]), fontsize=8)
ax.set_xlabel("hour_sin")
ax.set_ylabel("hour_cos")
ax.set_title("Cyclical Encoding of Hour")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 2. Spatial Features

In [ ]:
df_spat = add_spatial_features(df.copy())
spatial_cols = [c for c in df_spat.columns if c not in df.columns]
print(f"Spatial features added: {spatial_cols}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distance bucket distribution
if "distance_bucket" in df_spat.columns:
    df_spat["distance_bucket"].value_counts().sort_index().plot.bar(ax=axes[0], color="teal")
    axes[0].set_title("Distance Bucket Distribution")
    axes[0].set_xlabel("Bucket")
    axes[0].set_ylabel("Count")

# Bearing distribution
if "bearing" in df_spat.columns:
    axes[1].hist(df_spat["bearing"], bins=36, edgecolor="black", alpha=0.7)
    axes[1].set_xlabel("Bearing (degrees)")
    axes[1].set_title("Delivery Bearing Distribution")

plt.tight_layout()
plt.show()

## 3. Contextual & Interaction Features

In [ ]:
df_ctx = add_supply_demand_features(df.copy())
df_ctx = add_order_features(df_ctx)
ctx_cols = [c for c in df_ctx.columns if c not in df.columns]
print(f"Contextual features: {ctx_cols}")

# Add spatial + temporal for interaction features
df_full = add_temporal_features(df.copy())
df_full = add_spatial_features(df_full)
df_full = add_supply_demand_features(df_full)
df_full = add_order_features(df_full)
df_full = add_interaction_features(df_full)

interaction_cols = [c for c in df_full.columns if "_x_" in c]
print(f"\nInteraction features: {interaction_cols}")

## 4. Full Pipeline

In [ ]:
# Split data chronologically
train_df, val_df, test_df = split_by_time(df)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# Run pipeline
pipeline = FeaturePipeline()
X_train, y_train = pipeline.fit_transform(train_df)
X_val, y_val = pipeline.transform(val_df)
X_test, y_test = pipeline.transform(test_df)

print(f"\nFeature matrix shape: {X_train.shape}")
print(f"Features: {list(X_train.columns)}")

In [ ]:
# Feature correlation with target
correlations = X_train.assign(target=y_train).corr()["target"].drop("target").sort_values()

plt.figure(figsize=(10, 8))
correlations.plot.barh(color=["coral" if v < 0 else "steelblue" for v in correlations])
plt.xlabel("Pearson Correlation with Delivery Duration")
plt.title("Feature-Target Correlations")
plt.tight_layout()
plt.show()

## Summary

The feature pipeline produces **20+ numeric features** from raw order data:

| Category | Count | Top Feature |
|----------|-------|-------------|
| Temporal | 8 | `is_peak_dinner` |
| Spatial | 5 | `distance_km` |
| Contextual | 6 | `rider_utilization` |
| Interaction | 3+ | `distance_x_peak` |

Key design decisions:
- **Cyclical encoding** prevents discontinuity (hour 23 → 0)
- **Interaction features** capture non-linear combined effects
- **Pipeline class** ensures train/test consistency (same columns, same order)